In [ ]:
! rm -r sample_data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# %%capture
import os
from os.path import exists, join, basename, splitext

git_repo_url = 'https://github.com/CMU-Perceptual-Computing-Lab/openpose.git'
project_name = splitext(basename(git_repo_url))[0]
if not exists(project_name):
  # see: https://github.com/CMU-Perceptual-Computing-Lab/openpose/issues/949
  # install new CMake becaue of CUDA10
  !wget -q https://cmake.org/files/v3.13/cmake-3.13.0-Linux-x86_64.tar.gz
  !tar xfz cmake-3.13.0-Linux-x86_64.tar.gz --strip-components=1 -C /usr/local

  # clone openpose
  !git clone -q --depth 1 $git_repo_url
  # download models
  !wget -O /content/openpose/models/hand/pose_iter_102000.caffemodel https://polybox.ethz.ch/index.php/s/Oim76cuqrDVbdxm/download
  !wget -O /content/openpose/models/pose/body_25/pose_iter_584000.caffemodel https://polybox.ethz.ch/index.php/s/m5NQAhd7ukVPRoL/download
  !wget -O /content/openpose/models/face/pose_iter_116000.caffemodel https://polybox.ethz.ch/index.php/s/cEaF1FTpKjjJZbH/download
  !sed -i 's/execute_process(COMMAND git checkout master WORKING_DIRECTORY ${CMAKE_SOURCE_DIR}\/3rdparty\/caffe)/execute_process(COMMAND git checkout f019d0dfe86f49d1140961f8c7dec22130c83154 WORKING_DIRECTORY ${CMAKE_SOURCE_DIR}\/3rdparty\/caffe)/g' openpose/CMakeLists.txt
  # install system dependencies
  !apt-get -qq install -y libatlas-base-dev libprotobuf-dev libleveldb-dev libsnappy-dev libhdf5-serial-dev protobuf-compiler libgflags-dev libgoogle-glog-dev liblmdb-dev opencl-headers ocl-icd-opencl-dev libviennacl-dev
  # install python dependencies
  !pip install -q youtube-dl
  # build openpose
  !cd openpose && rm -rf build || true && mkdir build && cd build && cmake .. -DUSE_CUDNN=OFF && make -j`nproc`

--2024-04-27 01:48:48--  https://polybox.ethz.ch/index.php/s/Oim76cuqrDVbdxm/download
Resolving polybox.ethz.ch (polybox.ethz.ch)... 129.132.71.243
Connecting to polybox.ethz.ch (polybox.ethz.ch)|129.132.71.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 147344024 (141M) [application/octet-stream]
Saving to: ‘/content/openpose/models/hand/pose_iter_102000.caffemodel’

/content/openpose/m 100%[===================>] 140.52M  13.7MB/s    in 12s     

2024-04-27 01:49:01 (11.9 MB/s) - ‘/content/openpose/models/hand/pose_iter_102000.caffemodel’ saved [147344024/147344024]

--2024-04-27 01:49:01--  https://polybox.ethz.ch/index.php/s/m5NQAhd7ukVPRoL/download
Resolving polybox.ethz.ch (polybox.ethz.ch)... 129.132.71.243
Connecting to polybox.ethz.ch (polybox.ethz.ch)|129.132.71.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 104715850 (100M) [application/octet-stream]
Saving to: ‘/content/openpose/models/pose/body_25/pose_iter_584000

In [ ]:
%cd openpose

/content/openpose


In [ ]:
# !./build/examples/openpose/openpose.bin --video ./examples/media/video.avi --face --hand --write_images output_json_folder/ --display 0 --render_pose 0





In [ ]:
# !./build/examples/openpose/openpose.bin --video /content/drive/MyDrive/original/data_part_1-2-3/part_1/1.mp4 --face --write_images output_test_folder/ --display 0 --render_pose -1 --number_people_max 1 --disable_blending 1 --output_resolution 224x224 --net_resolution -1x256 --face_net_resolution 320x320





In [1]:
# import os
# import subprocess

# # OpenPose path
# openpose_bin_path = '/content/openpose/build/examples/openpose/openpose.bin'

# # Input directory
# root_dir = '/content/drive/MyDrive/original/archive/'

# # Output directory
# output_dir = '/content/drive/MyDrive/original/output/archive-output/'

# image_extensions = ['.jpg', '.png']

# # Make sure output is exist
# if not os.path.exists(output_dir):
#     os.makedirs(output_dir)

# def process_images(directory):
#     processed_count = 0
#     # Go throught images
#     for subdir, dirs, files in os.walk(directory):
#         for file in files:
#             if file.lower().endswith(tuple(image_extensions)):
#                 image_path = os.path.join(subdir, file)
#                 relative_path = os.path.relpath(subdir, directory)
#                 image_output_dir = os.path.join(output_dir, relative_path)

#                 if not os.path.exists(image_output_dir):
#                     os.makedirs(image_output_dir)

#                 image_output_path = os.path.join(image_output_dir, os.path.splitext(file)[0] + '_rendered.png')

#                 # If exist then skip
#                 if os.path.exists(image_output_path):
#                     print(f"Skipping {file}, output already exists.")
#                     continue

#                 # Construct OpenPose command
#                 command = f'{openpose_bin_path} --image "{image_path}" --write_images "{image_output_dir}" --display 0 --render_pose 1 --number_people_max 1 --disable_blending 1 --output_resolution 224x224 --net_resolution 160x-1 --face_net_resolution 320x320'
#                 print(f"Executing command: {command}")  # Debug: output running command

#                 try:
#                     # Run OpenPose command
#                     subprocess.run(command, shell=True, check=True)
#                     processed_count += 1
#                     print(f"Processed {file}: {processed_count}")
#                 except subprocess.CalledProcessError as e:
#                     print(f"Failed to process {file}. Error: {e}")
#     return processed_count

# # Process train ans test path
# total_train_processed = process_images(os.path.join(root_dir, 'train'))
# total_test_processed = process_images(os.path.join(root_dir, 'test'))

# print(f"Total processed files in train: {total_train_processed}")
# print(f"Total processed files in test: {total_test_processed}")


Total processed files in train: 0
Total processed files in test: 0


In [ ]:
import os

# 设置目录路径
root_dir_train = '/content/drive/MyDrive/archive/train/'
root_dir_test = '/content/drive/MyDrive/archive/test/'

# 列出文件
train_files = [f for f in os.listdir(root_dir_train) if os.path.isfile(os.path.join(root_dir_train, f))]
test_files = [f for f in os.listdir(root_dir_test) if os.path.isfile(os.path.join(root_dir_test, f))]

print("Train files:", train_files)
print("Test files:", test_files)


In [ ]:
import os
import subprocess

# OpenPose path
openpose_bin_path = '/content/openpose/build/examples/openpose/openpose.bin'

# Input directory
image_dir = '/content/drive/MyDrive/original/archive/'

# Output directory
output_dir = '/content/drive/MyDrive/original/output/archive-images'

# Support format
image_extensions = ['.jpg', '.jpeg', '.png']

# Get all images
def get_image_files(image_dir):
    image_files = []
    for root, _, files in os.walk(image_dir):
        for file in files:
            if os.path.splitext(file)[1].lower() in image_extensions:
                image_files.append(os.path.join(root, file))
    return image_files

image_files = get_image_files(image_dir)

# Create path
def create_output_dirs(output_dir, image_dir):
    for root, _, _ in os.walk(image_dir):
        relative_path = os.path.relpath(root, image_dir)
        output_subdir = os.path.join(output_dir, relative_path)
        if not os.path.exists(output_subdir):
            os.makedirs(output_subdir)

create_output_dirs(output_dir, image_dir)

# initial counter
processed_count = 0
counter = len(image_files)

# Go throught all images
for image_file in image_files:
    image_name, image_ext = os.path.splitext(image_file)
    relative_path = os.path.relpath(os.path.dirname(image_file), image_dir)
    output_image_path = os.path.join(output_dir, relative_path, f"{image_name}_pose{image_ext}")

    # If exist then skip
    if os.path.exists(output_image_path):
        print(f"Skipping {image_file}, output already exists.")
        counter -= 1
        continue

    # Construct OpenPose command
    command = f'{openpose_bin_path} --image "{image_file}" --face --write_images "{output_image_path}" --display 0'

    try:
        # Run OpenPose command
        subprocess.run(command, shell=True, check=True)
        processed_count += 1
        print(f"Processed {image_file}: {processed_count}/{counter}")
    except subprocess.CalledProcessError as e:
        print(f"Failed to process {image_file}. Error: {e}")

print(f"Total processed files: {processed_count}")
